# GRS Dataset Explorer

Interactive exploration of the GRS v3.0 dataset pool before sampling.

**Run order:** Execute cells top-to-bottom. Edit the config cell (Section 1) before running.

## Section 1 — Config & Data Loading

In [1]:
# ── USER CONFIG ─────────────────────────────────────────────
# Edit this cell before running the notebook.

DATASET_ROOT = "datasets"

SOURCE_MAP = {
    "grs_scenarios_v0.1": "GPT-5.2",
    "grs_scenarios_v0.2": "Gemini 2.5",
    "grs_scenarios_v0.3": "Claude Sonnet 4.6",
}

OBLIGATIONS_CONFIG = "configs/obligations.yaml"
# ─────────────────────────────────────────────────────────────

In [2]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="tab10")
SOURCE_ORDER = list(SOURCE_MAP.values())
COLORS = sns.color_palette("tab10", len(SOURCE_ORDER))

In [3]:
# ── DATA LOADING ─────────────────────────────────────────────
records = []
for version, source_name in SOURCE_MAP.items():
    path = Path(DATASET_ROOT) / version / "final" / "scenarios.jsonl"
    with open(path) as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                row["source"] = source_name
                records.append(row)

df = pd.DataFrame(records)

# Derived columns used throughout all sections
df["is_base"] = df["mutation_trace"].apply(
    lambda x: len(x.get("mutations", [])) == 0
)
df["mutation_families"] = df["mutation_trace"].apply(
    lambda x: [m["family"] for m in x.get("mutations", [])]
)
df["obligation_ids"] = df["seed_trace"].apply(
    lambda x: x.get("obligation_ids", [])
)

# Sanity check
total = len(df)
print(f"Loaded {total} scenarios total")
for source_name in SOURCE_ORDER:
    count = len(df[df["source"] == source_name])
    print(f"  {source_name}: {count} scenarios ({count / total * 100:.1f}%)")


Loaded 2004 scenarios total
  GPT-5.2: 570 scenarios (28.4%)
  Gemini 2.5: 641 scenarios (32.0%)
  Claude Sonnet 4.6: 793 scenarios (39.6%)
